# 面试问题：多副本 LLM 服务怎样兼顾 Prefix KV Cache、队列、容量、隔离与公平来路由？

**一句话回答。** 先用模型版本、Tokenizer、Adapter 和租户安全域筛掉不兼容副本，再估算每个副本可复用的块对齐前缀、排队时间和本次请求需要新增的 KV；容量不足的副本直接拒绝，剩余副本按“复用收益减去排队、容量压力和同租户热点惩罚”排序。跨租户公平应在请求准入层用加权虚拟时间实现，选定副本后还要做带版本的原子预留，完成时释放容量、更新 Trie 和指标。

只选最长前缀会把请求持续压向一个热点副本；只选最短队列又会浪费已经计算好的 KV。正确做法是把 locality 当作可量化收益，而不是不可打破的亲和性，并把安全隔离与容量硬约束放在软打分之前。

本 Notebook 用标准 Python 从零实现一个小型前缀 Trie 和路由状态机，不调用推理服务框架，也不声称教学参数等价于线上吞吐。思路参考 [Preble](https://arxiv.org/abs/2407.00023)：其核心问题正是分布式 LLM Serving 中如何利用 prefix cache locality，同时处理负载不均。

In [ ]:
from collections import defaultdict  # 导入带默认值的字典，用于维护租户虚拟时间和计数。
question = "多副本 LLM 的前缀缓存感知路由"  # 保存本实验要回答的核心面试问题。
design_goals = ("缓存局部性", "队列负载", "KV容量", "租户隔离", "加权公平")  # 列出路由策略必须同时处理的五类目标。
assert "前缀" in question  # 确认题目明确讨论 Prefix KV Cache。
assert len(design_goals) == 5  # 确认设计目标没有遗漏关键维度。
assert "租户隔离" in design_goals  # 确认安全隔离被视为一等约束。
assert "加权公平" in design_goals  # 确认公平不是仅靠平均吞吐间接推断。

## 1. Prefix 命中必须绑定完整缓存身份

KV 不能仅按提示词字符串共享。有效缓存键至少包含精确 token 序列、模型权重版本、Tokenizer 版本、Adapter，以及租户或显式授权的共享域；任一字段变化都可能让旧 KV 无效或造成数据越权。Paged KV 往往按 block 管理，因此教学 Trie 只承认完整块边界，七个 token 的逻辑前缀在 block size 为二时只能复用六个。

Trie 的价值是让路由器用提示词 token 逐层查找最长已物化前缀，而不需要扫描副本上的全部缓存条目。下面把租户和制品指纹放在不同根节点，天然阻止另一个租户仅因 token 相同而取得命中。

In [ ]:
class PrefixTrie:  # 定义一个按租户和制品指纹隔离的小型前缀 Trie。
    def __init__(self, block_size=2):  # 初始化 KV block 大小和所有隔离根节点。
        self.block_size = block_size  # 保存块大小，使缓存命中只能落在完整块边界。
        self.roots = {}  # 用二元身份键保存互相隔离的 Trie 根节点。
    def insert(self, tenant, artifact, tokens, cached_tokens):  # 写入某租户某制品已经物化的 token 前缀。
        aligned = min(len(tokens), cached_tokens // self.block_size * self.block_size)  # 将可缓存长度向下对齐到完整 KV block。
        node = self.roots.setdefault((tenant, artifact), {})  # 为租户和制品组合取得独立根节点。
        for index, token in enumerate(tokens[:aligned], start=1):  # 沿着实际已缓存的 token 前缀逐层创建节点。
            node = node.setdefault(token, {})  # 取得或创建当前 token 对应的子节点。
            if index % self.block_size == 0: node["_cached_tokens"] = index  # 只在完整 block 结束处记录可复用长度。
    def longest(self, tenant, artifact, tokens):  # 查询请求在指定隔离域中的最长块对齐命中。
        node = self.roots.get((tenant, artifact), {})  # 找不到身份完全一致的根节点时从空 Trie 开始。
        best = 0  # 初始化最长可复用 token 数为零。
        for token in tokens:  # 按请求 token 顺序向下遍历 Trie。
            if token not in node: break  # 一旦 token 不匹配就停止，后缀不能越过断点复用。
            node = node[token]  # 进入匹配 token 的下一层节点。
            best = node.get("_cached_tokens", best)  # 仅在块边界标记存在时推进最长命中。
        return best  # 返回可以安全复用的块对齐 token 数。
artifact = ("model-v3", "base", "tok-v2")  # 构造模型、Adapter 与 Tokenizer 的缓存制品指纹。
prompt_tokens = tuple("系统 你 是 助手 用户 请 总结 缓存 路由".split())  # 用确定 token 序列模拟一个九 token 提示词。
demo_trie = PrefixTrie(block_size=2)  # 创建每两个 token 一个 KV block 的教学 Trie。
demo_trie.insert("alpha", artifact, prompt_tokens, 7)  # 为 alpha 写入七 token 逻辑前缀，其中六 token 可整块复用。
demo_trie.insert("beta", artifact, prompt_tokens, 8)  # 为 beta 在独立根节点写入八 token 前缀。
assert demo_trie.longest("alpha", artifact, prompt_tokens) == 6  # 验证七 token 被向下对齐为六 token 命中。
assert demo_trie.longest("beta", artifact, prompt_tokens) == 8  # 验证 beta 的独立缓存可命中八 token。
assert demo_trie.longest("gamma", artifact, prompt_tokens) == 0  # 验证未登记租户无法借用相同文本的缓存。
assert demo_trie.longest("alpha", ("model-v4", "base", "tok-v2"), prompt_tokens) == 0  # 验证模型版本变化会使旧 KV 失效。
assert demo_trie.longest("alpha", artifact, prompt_tokens[:-4] + ("改写",)) == 4  # 验证 token 分叉后只返回分叉前的完整块。

## 2. 路由依据必须来自带版本的副本快照

控制面需要周期性收集每个副本的可用 KV、总容量、保留水位、排队工作量、处理速率、当前租户并发量和缓存 Trie。快照必须带 version；从打分到预留之间若状态变化，旧计划不能继续扣减容量。

示例中 `r1` 对目标租户有更长前缀但已有少量排队，`r2` 更空闲但只能复用一个 block，`r3` 仍运行旧模型版本。允许租户集合是硬授权条件，不参与“分高就放行”的软排序。

In [ ]:
def make_replica(name, artifact_value, kv_free, queued_tokens, allowed_tenants):  # 构造一个带版本和独立 Trie 的副本快照。
    return {"id": name, "artifact": artifact_value, "kv_capacity": 100, "kv_free": kv_free, "kv_reserve": 10, "queued_tokens": queued_tokens, "token_rate": 20.0, "allowed_tenants": set(allowed_tenants), "tenant_inflight": defaultdict(int), "trie": PrefixTrie(2), "version": 1, "state": "ready"}  # 返回路由所需的全部最小状态。
r1 = make_replica("r1", artifact, 60, 8, {"alpha", "beta"})  # 创建前缀较热且有少量排队的副本。
r2 = make_replica("r2", artifact, 90, 0, {"alpha", "beta"})  # 创建缓存较少但队列和容量更宽松的副本。
r3 = make_replica("r3", ("model-v2", "base", "tok-v2"), 95, 0, {"alpha"})  # 创建仍运行旧模型版本的不兼容副本。
r1["trie"].insert("alpha", artifact, prompt_tokens, 6)  # 在 r1 上登记 alpha 的三个完整前缀块。
r1["trie"].insert("beta", artifact, prompt_tokens, 8)  # 在 r1 的独立租户域登记 beta 的四个块。
r2["trie"].insert("alpha", artifact, prompt_tokens, 2)  # 在 r2 上只登记 alpha 的一个前缀块。
r1["tenant_inflight"]["alpha"] = 1  # 模拟 alpha 已经占用 r1 的一个并发槽位。
assert r1["trie"].longest("alpha", artifact, prompt_tokens) == 6  # 验证 r1 对目标租户具有较长缓存命中。
assert r2["trie"].longest("alpha", artifact, prompt_tokens) == 2  # 验证 r2 只具有较短缓存命中。
assert r1["trie"].longest("gamma", artifact, prompt_tokens) == 0  # 验证副本内不同租户仍严格隔离。
assert r3["artifact"][0] == "model-v2"  # 确认第三个副本会因旧模型版本被兼容性过滤。

## 3. 请求合同决定兼容性和资源上界

路由请求应携带模型修订、Adapter、Tokenizer、租户、输入 token、最大输出 token 和前缀域。最大输出长度不是生成承诺，却是容量预留上界；若只按 prompt 长度准入，解码阶段可能耗尽 KV。相同文本在不同 Tokenizer 下产生的 token 不同，因此不能跨 Tokenizer 复用。

私有前缀域要求租户完全一致。若产品支持公共系统提示词共享，应由离线构建的只读公共域显式标记，而不是让运行时根据“内容看起来一样”自动跨租户合并。

In [ ]:
def request_artifact(request_value):  # 从请求中提取必须精确匹配的缓存制品指纹。
    return (request_value["model_revision"], request_value["adapter"], request_value["tokenizer"])  # 按固定顺序返回模型、Adapter 和 Tokenizer。
def is_compatible(replica, request_value):  # 判断副本是否满足不可通过打分绕过的硬约束。
    return replica["state"] == "ready" and replica["artifact"] == request_artifact(request_value) and request_value["tenant"] in replica["allowed_tenants"] and request_value["prefix_scope"] == "private"  # 同时检查运行状态、制品、授权租户和隔离域。
request = {"id": "q-alpha-7", "tenant": "alpha", "model_revision": "model-v3", "adapter": "base", "tokenizer": "tok-v2", "prefix_scope": "private", "tokens": prompt_tokens, "max_output_tokens": 12, "prefill_cost": 0.4, "tenant_weight": 1.0}  # 构造一个带资源上界和租户权重的请求合同。
wrong_tokenizer = {**request, "tokenizer": "tok-v3"}  # 构造 Tokenizer 不匹配的请求用于反例。
intruder = {**request, "tenant": "gamma"}  # 构造未获副本授权的租户请求用于反例。
assert request_artifact(request) == artifact  # 验证请求制品指纹与两个新版本副本一致。
assert is_compatible(r1, request)  # 验证 r1 通过全部硬兼容约束。
assert is_compatible(r2, request)  # 验证 r2 也通过全部硬兼容约束。
assert not is_compatible(r3, request)  # 验证旧模型副本不会因空闲而被错误选中。
assert not is_compatible(r1, wrong_tokenizer)  # 验证 Tokenizer 变化禁止复用和路由。
assert not is_compatible(r1, intruder)  # 验证未授权租户不能依靠高分绕过隔离。

## 4. 把每个候选副本拆成可解释的成本项

对于兼容副本，先计算 `cached_tokens`，再得到 `new_kv = uncached_prompt + max_output`。若扣除新增 KV 后低于保留水位，候选立即失效；保留水位用于吸收长度估计误差、并发完成延迟和系统块碎片。

软分数由四部分构成：复用前缀节省的 prefill 成本、预计排队延迟、剩余容量低于目标比例时的压力惩罚，以及该租户已经占用该副本时的热点惩罚。线上系数应来自压测与遥测校准，不能把下面的教学常数直接复制到生产。

In [ ]:
def estimate_candidate(replica, request_value):  # 将一个副本拆解为缓存、队列、容量和热点四类成本。
    artifact_value = request_artifact(request_value)  # 取得请求要求的精确制品指纹。
    cached = replica["trie"].longest(request_value["tenant"], artifact_value, request_value["tokens"]) if is_compatible(replica, request_value) else 0  # 仅为硬约束兼容的副本查询租户私有前缀。
    uncached = len(request_value["tokens"]) - cached  # 计算仍需执行 prefill 的输入 token 数。
    required_kv = uncached + request_value["max_output_tokens"]  # 按未命中输入与最大输出估计最坏 KV 增量。
    post_free = replica["kv_free"] - required_kv  # 预测接纳请求后的剩余 KV token 容量。
    queue_delay = replica["queued_tokens"] / replica["token_rate"]  # 用排队 token 工作量除以处理速率估计等待时间。
    saved_prefill = cached * request_value["prefill_cost"]  # 把缓存命中换算成节省的 prefill 成本单位。
    free_ratio = post_free / replica["kv_capacity"]  # 计算接纳后的 KV 空闲比例。
    pressure_penalty = max(0.0, 0.2 - free_ratio) * 10.0  # 低于百分之二十目标水位时施加非线性前的教学惩罚。
    hotspot_penalty = replica["tenant_inflight"][request_value["tenant"]] / request_value["tenant_weight"] * 0.25  # 惩罚同租户继续堆积在同一副本。
    eligible = is_compatible(replica, request_value) and post_free >= replica["kv_reserve"]  # 将授权、制品和保留水位作为硬准入条件。
    score = saved_prefill - queue_delay - pressure_penalty - hotspot_penalty  # 得到可解释的综合收益分数。
    return {"replica": replica["id"], "snapshot_version": replica["version"], "cached_tokens": cached, "required_kv": required_kv, "post_free": post_free, "queue_delay": queue_delay, "eligible": eligible, "score": score}  # 返回可记录和重放的候选计划。
e1 = estimate_candidate(r1, request)  # 估算具有长前缀和小队列的 r1。
e2 = estimate_candidate(r2, request)  # 估算具有短前缀和空队列的 r2。
e3 = estimate_candidate(r3, request)  # 估算旧模型副本以验证硬过滤。
assert e1["cached_tokens"] == 6  # 验证 r1 可复用六个输入 token。
assert e1["required_kv"] == 15  # 验证 r1 只需三个 prefill KV 加十二个输出 KV。
assert e2["required_kv"] == 19  # 验证 r2 因命中较短而需要更多新 KV。
assert e1["post_free"] == 45  # 验证容量预测正确扣除了最坏 KV 增量。
assert e1["eligible"] and e2["eligible"] and not e3["eligible"]  # 验证新版本副本可选而旧版本副本被排除。

## 5. Locality 是收益，不是绝对亲和性

初始状态下，`r1` 多复用四个 token 的收益足以覆盖小队列，所以它胜出；当同一副本排队工作量显著上升时，路由应改选 `r2`。这就是 locality-aware 而非 locality-only。

排序还必须可重复：分数相同时使用副本 ID 打破平局，便于重放和审计。旧模型、未授权租户和容量不足的副本不会出现在候选列表里，而不是以一个很低的分数侥幸参与竞争。

In [ ]:
def rank_candidates(replicas, request_value):  # 对通过硬约束的副本进行确定性的综合收益排序。
    plans = [estimate_candidate(replica, request_value) for replica in replicas]  # 为每个副本生成一份可解释候选计划。
    return sorted((plan for plan in plans if plan["eligible"]), key=lambda plan: (-plan["score"], plan["replica"]))  # 先按分数降序，再按副本 ID 稳定打破平局。
initial_ranking = rank_candidates([r1, r2, r3], request)  # 在初始快照上执行缓存感知路由。
overloaded_r1 = {**r1, "queued_tokens": 80}  # 构造 r1 排队工作量显著增大的新快照。
overload_ranking = rank_candidates([overloaded_r1, r2, r3], request)  # 在热点状态下重新计算而不固守缓存亲和性。
pressured_r1 = {**r1, "kv_free": 20}  # 构造接纳后会跌破十 token 保留水位的 r1。
pressure_ranking = rank_candidates([pressured_r1, r2, r3], request)  # 验证容量硬约束优先于长前缀收益。
assert [plan["replica"] for plan in initial_ranking] == ["r1", "r2"]  # 验证小队列时长前缀 r1 位于首位。
assert overload_ranking[0]["replica"] == "r2"  # 验证队列过载后请求转移到缓存较少的 r2。
assert pressure_ranking[0]["replica"] == "r2"  # 验证 r1 容量不足时不会参与软分竞争。
assert all(plan["replica"] != "r3" for plan in initial_ranking)  # 验证旧模型副本从候选集合彻底消失。
assert initial_ranking[0]["cached_tokens"] > overload_ranking[0]["cached_tokens"]  # 验证策略允许为负载均衡主动牺牲部分 locality。

## 6. 副本路由不能代替跨租户公平调度

“这个请求去哪个副本”和“下一个服务哪个租户”是两个层次。若总是先处理缓存命中最高的租户，长公共前缀的热门租户可能饿死其他租户。教学调度器为每个租户维护加权虚拟时间，以 `当前虚拟时间 + 预计 token 工作量 / 权重` 作为虚拟完成时间，优先选择最小者。

权重表达购买的服务份额，虚拟时间表达已经消费的份额；它们不应混入缓存身份。公平层先选择请求，副本层再为该请求寻找最低综合成本，两层指标需要分别观测。

In [ ]:
def choose_fair_request(pending, virtual_time, weights):  # 用加权虚拟完成时间选择下一租户请求。
    def virtual_finish(request_value):  # 定义单个请求消费公平份额后的虚拟完成时间。
        work = len(request_value["tokens"]) + request_value["max_output_tokens"]  # 用输入加最大输出构造保守 token 工作量。
        return virtual_time[request_value["tenant"]] + work / weights[request_value["tenant"]]  # 按租户权重折算新增虚拟服务时间。
    return min(pending, key=lambda request_value: (virtual_finish(request_value), request_value["id"])), virtual_finish  # 返回最早虚拟完成的请求和可审计估值函数。
beta_request = {**request, "id": "q-beta-3", "tenant": "beta", "tenant_weight": 2.0}  # 构造具有双倍服务权重的 beta 请求。
virtual_time = defaultdict(float, {"alpha": 4.0, "beta": 1.0})  # 记录 alpha 已消费更多历史份额的调度状态。
tenant_weights = {"alpha": 1.0, "beta": 2.0}  # 声明两个租户约定的长期服务权重。
chosen_request, finish_value = choose_fair_request([request, beta_request], virtual_time, tenant_weights)  # 在副本路由前先做跨租户公平选择。
virtual_time[chosen_request["tenant"]] = finish_value(chosen_request)  # 将被服务租户的虚拟时间推进到本次完成位置。
assert chosen_request["tenant"] == "beta"  # 验证历史消费更少且权重更高的 beta 先获得服务。
assert finish_value(beta_request) < finish_value(request)  # 验证选择结果对应更小的虚拟完成时间。
assert virtual_time["alpha"] == 4.0  # 验证未被服务租户的虚拟时间不会被错误推进。
assert virtual_time["beta"] > 1.0  # 验证已服务租户会记录本次资源消费。
assert r1["trie"].longest("beta", artifact, prompt_tokens) == 8  # 验证公平身份与缓存身份独立且仍按租户查询。

## 7. 计划和预留之间要防止 TOCTOU

打分结果只是 `planned` 状态，包含目标副本、快照版本和预估 KV。真正准入时必须在副本锁或单线程调度循环内重新检查版本、容量与计划状态，然后原子扣减 KV 并转为 `reserved`。若版本已变化，就返回 `retry`，重新采样，而不是相信几毫秒前的结果。

真实系统还要为 reservation 设置超时：请求在预留后未开始执行时自动回收；调度器崩溃后根据持久化请求 ID 或租约重建状态，避免永久泄漏 KV 配额。

In [ ]:
def reserve_plan(plan, replica):  # 将计划按快照版本原子转成容量预留。
    if plan["state"] != "planned": return False  # 拒绝重复预留或已经进入终态的计划。
    if replica["version"] != plan["snapshot_version"]: plan["state"] = "retry"; return False  # 快照变化时标记重试以消除检查与使用竞态。
    if replica["kv_free"] - plan["required_kv"] < replica["kv_reserve"]: plan["state"] = "retry"; return False  # 二次检查当前容量仍高于保留水位。
    replica["kv_free"] -= plan["required_kv"]  # 原子扣减该请求的最坏 KV 配额。
    replica["tenant_inflight"][plan["tenant"]] += 1  # 增加目标副本上该租户的并发计数。
    replica["version"] += 1  # 推进副本版本，使其他旧计划在预留时失效。
    plan["reservation_version"] = replica["version"]  # 记录成功预留后的副本版本以便完成阶段审计。
    plan["state"] = "reserved"  # 将状态从 planned 推进到 reserved。
    return True  # 向调用方报告原子预留成功。
active_r1 = {**r1, "tenant_inflight": defaultdict(int, r1["tenant_inflight"])}  # 复制可变计数，避免教学预留污染先前快照。
best = initial_ranking[0]  # 取得初始排序中收益最高的 r1 候选。
plan = {**best, "request_id": request["id"], "tenant": request["tenant"], "state": "planned"}  # 把候选估值封装成带状态的执行计划。
free_before = active_r1["kv_free"]  # 保存预留前容量供后置条件验证。
assert reserve_plan(plan, active_r1)  # 验证版本和容量一致时可以成功预留。
assert plan["state"] == "reserved"  # 验证计划进入 reserved 状态。
assert active_r1["kv_free"] == free_before - plan["required_kv"]  # 验证预留精确扣减估算 KV。
assert active_r1["tenant_inflight"]["alpha"] == 2  # 验证同租户并发计数随预留增加。
stale_plan = {**best, "request_id": "q-stale", "tenant": "alpha", "state": "planned"}  # 复用旧快照版本构造陈旧计划。
assert not reserve_plan(stale_plan, active_r1)  # 验证副本版本变化后旧计划无法继续执行。
assert stale_plan["state"] == "retry"  # 验证陈旧计划被显式送回重新路由状态。

## 8. 完成回写闭合反馈环，并用多维指标验收

请求结束后释放预留的最坏情况容量，再把实际保留的 prompt block 写回对应租户 Trie。失败请求是否保留已算出的前缀取决于引擎语义与淘汰策略；示例只在成功时写回。更新必须同时减少租户并发、推进副本版本，并把计划转成终态。

评测至少报告 cache reuse ratio、排队延迟分位数、KV 拒绝率、跨租户误复用数、各租户吞吐与 Jain 公平指数。平均吞吐很高并不能证明公平或安全；路由策略还应在热点前缀、突发长输出、模型滚动升级、Adapter 切换和副本故障场景下回放。

In [ ]:
def finish_plan(plan, replica, request_value, success):  # 完成请求并释放容量、更新缓存和租户占用。
    if plan["state"] != "reserved": return False  # 只有已预留计划能够进入完成路径。
    replica["kv_free"] = min(replica["kv_capacity"], replica["kv_free"] + plan["required_kv"])  # 释放预留的最坏 KV 配额并限制在总容量内。
    replica["tenant_inflight"][plan["tenant"]] = max(0, replica["tenant_inflight"][plan["tenant"]] - 1)  # 安全减少对应租户的副本并发数。
    if success: replica["trie"].insert(plan["tenant"], request_artifact(request_value), request_value["tokens"], len(request_value["tokens"]))  # 成功时把实际 prompt 完整块写回私有 Trie。
    replica["version"] += 1  # 推进副本版本以反映释放和缓存目录变化。
    plan["state"] = "completed" if success else "failed"  # 根据执行结果写入唯一终态。
    return True  # 报告完成回写已执行。
assert finish_plan(plan, active_r1, request, True)  # 验证成功请求能够闭合预留生命周期。
assert plan["state"] == "completed"  # 验证计划进入 completed 终态。
assert active_r1["kv_free"] == free_before  # 验证最坏情况预留在完成后完整释放。
assert active_r1["trie"].longest("alpha", artifact, prompt_tokens) == 8  # 验证九 token prompt 只新增到八 token 完整块边界。
assert active_r1["trie"].longest("gamma", artifact, prompt_tokens) == 0  # 验证完成回写没有污染其他租户域。
served_tokens = {"alpha": 100.0, "beta": 100.0}  # 构造两个同权租户的观测吞吐用于公平性教学。
jain_index = sum(served_tokens.values()) ** 2 / (len(served_tokens) * sum(value ** 2 for value in served_tokens.values()))  # 计算 Jain 公平指数作为长期份额指标。
route_metrics = {"reused_tokens": e1["cached_tokens"], "cross_tenant_reuse": 0, "kv_rejections": 1, "jain_index": jain_index}  # 汇总缓存、安全、容量和公平四类指标。
assert route_metrics["reused_tokens"] == 6  # 验证指标记录本次计划实际估算的复用量。
assert route_metrics["cross_tenant_reuse"] == 0  # 验证安全指标没有跨租户误复用。
assert route_metrics["kv_rejections"] == 1  # 验证容量不足候选被单独计数而非静默丢弃。
assert route_metrics["jain_index"] == 1.0  # 验证相同服务份额对应完全公平的教学基线。

## 面试总结

可以按五层回答这个问题：第一层用模型、Tokenizer、Adapter 和租户域定义缓存身份；第二层用块对齐 Trie 估算真实可复用前缀；第三层把 KV 容量与授权作为硬约束，再权衡 prefill 收益、队列和热点；第四层用加权虚拟时间保证跨租户公平；第五层用版本化 plan/reserve/finish 状态机消除竞态并回写缓存。

进一步追问时要指出：路由器只是在预测，真实服务时间还受 batch 形状、prefill/decode 分离、网络拓扑、KV 碎片和抢占影响；公平也不等于所有租户吞吐相等，而是按约定权重长期分享资源。生产实现需要一致性哈希或分层目录降低 Trie 元数据成本、租约与故障转移、隐私审计，以及基于真实 trace 的系数校准。